In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class MultiLabelImageFolder(Dataset):
    """Кастомный Dataset для multi-label классификации"""
    def __init__(self, data_path, transform=None, label_file=None):
        self.data_path = data_path
        self.transform = transform
        
        self.image_paths = []
        self.labels = []
        self.classes = []
        
        if label_file and os.path.exists(label_file):
            self._load_from_csv(label_file)
        else:
            self._load_from_folders()
    
    def _load_from_folders(self):
        """Загрузка данных из структуры папок (multi-label)"""
        self.classes = sorted([d for d in os.listdir(self.data_path) 
                             if os.path.isdir(os.path.join(self.data_path, d))])
        
        class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        image_dict = {}
        
        for class_idx, class_name in enumerate(self.classes):
            class_path = os.path.join(self.data_path, class_name)
            if not os.path.isdir(class_path):
                continue
                
            for img_name in os.listdir(class_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_path, img_name)
                    
                    if img_path not in image_dict:
                        image_dict[img_path] = []
                    
                    image_dict[img_path].append(class_idx)
        
        self.image_paths = list(image_dict.keys())
        self.labels = [self._to_multi_hot(labels, len(self.classes)) 
                      for labels in image_dict.values()]
        
        print(f"Найдено {len(self.image_paths)} изображений с {len(self.classes)} классами")
    
    def _load_from_csv(self, label_file):
        """Загрузка данных из CSV файла с multi-label метками"""
        df = pd.read_csv(label_file)
        self.classes = [col for col in df.columns if col not in ['image', 'filename']]
        
        for _, row in df.iterrows():
            img_name = row['image'] if 'image' in row else row['filename']
            img_path = os.path.join(self.data_path, img_name)
            
            if os.path.exists(img_path):
                self.image_paths.append(img_path)
                labels = [int(row[cls]) for cls in self.classes]
                self.labels.append(labels)
    
    def _to_multi_hot(self, indices, num_classes):
        """Конвертирует список индексов в multi-hot вектор"""
        multi_hot = [0] * num_classes
        for idx in indices:
            if idx < num_classes:
                multi_hot[idx] = 1
        return multi_hot
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.float32)
        return image, label

def load_and_split_data(data_path, train_ratio=0.8, label_file=None):
    """Загружает данные для multi-label классификации и разделяет на выборки"""
    print(f"Загрузка multi-label данных из: {data_path}")
    
    if not os.path.exists(data_path):
        print(f"Ошибка: Папка {data_path} не существует!")
        return None, None, None
    
    full_dataset = MultiLabelImageFolder(
        data_path=data_path,
        transform=train_transform,
        label_file=label_file
    )
    
    print(f"Найдено классов: {full_dataset.classes}")
    print(f"Всего изображений: {len(full_dataset)}")
    
    train_size = int(train_ratio * len(full_dataset))
    test_size = len(full_dataset) - train_size
    
    print(f"Тренировочные данные: {train_size} изображений")
    print(f"Тестовые данные: {test_size} изображений")
    
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
    
    test_dataset_full = MultiLabelImageFolder(
        data_path=data_path,
        transform=val_transform,
        label_file=label_file
    )
    test_dataset = torch.utils.data.Subset(test_dataset_full, test_dataset.indices)
    
    return train_dataset, test_dataset, full_dataset.classes

class MultiLabelLoss(nn.Module):
    """Loss функция для multi-label классификации"""
    def __init__(self):
        super(MultiLabelLoss, self).__init__()
        self.bce_loss = nn.BCEWithLogitsLoss()
    
    def forward(self, outputs, targets):
        return self.bce_loss(outputs, targets)

def calculate_multi_label_accuracy(outputs, targets, threshold=0.5):
    """Вычисляет accuracy для multi-label классификации"""
    with torch.no_grad():
        preds = torch.sigmoid(outputs) > threshold
        correct = (preds == targets.bool()).float()
        
        exact_match = (correct.sum(dim=1) == targets.size(1)).float().mean()
        label_accuracy = correct.mean()
        
        return exact_match.item(), label_accuracy.item()

class ResNetWithBBox(nn.Module):
    """
    ResNet модель с добавлением параметров bounding box к эмбеддингам
    перед классификационной частью
    """
    def __init__(self, num_classes, pretrained=True, bbox_dim=4):
        super(ResNetWithBBox, self).__init__()
        
        self.resnet = models.resnet18(pretrained=pretrained)
        
        self.original_fc_in_features = self.resnet.fc.in_features
        
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-1])
        
        self.bbox_dim = bbox_dim
        self.classifier = nn.Linear(
            self.original_fc_in_features + bbox_dim, 
            num_classes
        )
        
        nn.init.normal_(self.classifier.weight, 0, 0.01)
        nn.init.constant_(self.classifier.bias, 0)
        
        print(f"Модель инициализирована:")
        print(f"  - Входные признаки ResNet: {self.original_fc_in_features}")
        print(f"  - BBox параметры: {bbox_dim}")
        print(f"  - Выходные классы: {num_classes}")
    
    def forward(self, x, bbox_params=None):
        """
        Forward pass с дополнительными bbox параметрами
        
        Args:
            x: входные изображения [batch_size, 3, H, W]
            bbox_params: тензор bbox параметров [batch_size, 4]
                       формат: [x_center, y_center, width, height] (нормированные 0-1)
        """
        features = self.resnet(x)
        features = features.view(features.size(0), -1)
        
        if bbox_params is None:
            batch_size = features.size(0)
            bbox_params = torch.zeros(batch_size, self.bbox_dim).to(features.device)
        
        combined_features = torch.cat([features, bbox_params], dim=1)
        
        output = self.classifier(combined_features)
        
        return output

def create_resnet_with_bbox(num_classes, pretrained=True):
    """Создает модель ResNet с поддержкой bbox параметров"""
    return ResNetWithBBox(num_classes, pretrained)

def prepare_bbox_params(bbox_tensor, img_size=(224, 224)):
    """
    Подготавливает bbox параметры для модели
    
    Args:
        bbox_tensor: тензор с bbox в формате [x1, y1, w, h] или [x_center, y_center, w, h]
        img_size: размер изображения (H, W) для нормализации
        
    Returns:
        Нормированные bbox параметры [x_center, y_center, width, height] в диапазоне [0, 1]
    """
    if bbox_tensor.dim() == 1:
        bbox_tensor = bbox_tensor.unsqueeze(0)
    
    img_h, img_w = img_size
    bbox_normalized = bbox_tensor.clone().float()
    
    if bbox_normalized.size(1) == 4:
        x1, y1, w, h = bbox_normalized[:, 0], bbox_normalized[:, 1], bbox_normalized[:, 2], bbox_normalized[:, 3]
        x_center = (x1 + w/2) / img_w
        y_center = (y1 + h/2) / img_h
        width_norm = w / img_w
        height_norm = h / img_h
    else:
        x_center, y_center, width_norm, height_norm = bbox_normalized[:, 0], bbox_normalized[:, 1], bbox_normalized[:, 2], bbox_normalized[:, 3]
        x_center = x_center / img_w
        y_center = y_center / img_h
        width_norm = width_norm / img_w
        height_norm = height_norm / img_h
    
    bbox_params = torch.stack([x_center, y_center, width_norm, height_norm], dim=1)
    
    bbox_params = torch.clamp(bbox_params, 0.0, 1.0)
    
    return bbox_params

def yolo_to_bbox_params(yolo_detection, img_size=(224, 224)):
    """
    Конвертирует детекцию YOLO в bbox параметры для ResNet
    
    Args:
        yolo_detection: результат YOLO детекции
        img_size: размер изображения
        
    Returns:
        bbox_params: тензор [x_center, y_center, width, height] нормированный [0, 1]
    """
    if len(yolo_detection.boxes) == 0:
        return torch.tensor([0.5, 0.5, 0.1, 0.1]).float()
    
    box = yolo_detection.boxes[0]
    xywh = box.xywh[0].cpu().numpy()  # [x_center, y_center, width, height]
    
    bbox_tensor = torch.tensor(xywh).float()
    bbox_params = prepare_bbox_params(bbox_tensor, img_size)
    
    return bbox_params.squeeze(0)

def train_model_with_bbox(model, train_loader, val_loader, criterion, optimizer, epochs=25, bbox_provider=None):
    """
    Обучение модели с bbox параметрами
    
    Args:
        bbox_provider: функция, которая возвращает bbox параметры для batch
    """
    model = model.to(device)
    train_losses = []
    val_accuracies = []
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            if bbox_provider is not None:
                bbox_params = bbox_provider(images, labels)
                bbox_params = bbox_params.to(device)
                outputs = model(images, bbox_params)
            else:
                outputs = model(images)
            
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            
            if batch_idx % 10 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}')
        
        model.eval()
        total_exact_accuracy = 0.0
        total_label_accuracy = 0.0
        val_batches = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                if bbox_provider is not None:
                    bbox_params = bbox_provider(images, labels)
                    bbox_params = bbox_params.to(device)
                    outputs = model(images, bbox_params)
                else:
                    outputs = model(images)
                
                exact_acc, label_acc = calculate_multi_label_accuracy(outputs, labels)
                total_exact_accuracy += exact_acc
                total_label_accuracy += label_acc
                val_batches += 1
        
        epoch_loss = running_loss / len(train_loader)
        epoch_exact_accuracy = 100 * total_exact_accuracy / val_batches
        epoch_label_accuracy = 100 * total_label_accuracy / val_batches
        
        train_losses.append(epoch_loss)
        val_accuracies.append(epoch_label_accuracy)
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}')
        print(f'Exact Match Accuracy: {epoch_exact_accuracy:.2f}%')
        print(f'Label Accuracy: {epoch_label_accuracy:.2f}%')
        print('-' * 50)
    
    return train_losses, val_accuracies

class YOLOBBoxProvider:
    """Провайдер bbox параметров используя YOLO модель"""
    def __init__(self, yolo_model_path, img_size=(224, 224)):
        self.yolo_model = YOLO(yolo_model_path)
        self.img_size = img_size
    
    def __call__(self, images, labels=None):
        """
        Для batch изображений возвращает bbox параметры
        """
        batch_size = images.size(0)
        bbox_params_list = []
        
        for i in range(batch_size):
            img_tensor = images[i]
            img_np = (img_tensor.cpu().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
            img_pil = Image.fromarray(img_np)
            
            results = self.yolo_model(img_pil)
            
            if len(results) > 0 and len(results[0].boxes) > 0:
                bbox_params = yolo_to_bbox_params(results[0], self.img_size)
            else:
                bbox_params = torch.tensor([0.5, 0.5, 0.1, 0.1]).float()
            
            bbox_params_list.append(bbox_params)
        
        return torch.stack(bbox_params_list)

def check_data_structure():
    """Проверяет существование различных возможных путей"""
    possible_paths = [
        "../merged_augmented_data",
        "merged_augmented_data", 
        "./merged_augmented_data",
        r"C:\Users\user\Desktop\MIPT_year_2\engineer_workshop\Working_directory\Processed_data\merged_augmented_data"
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            print(f"Найдена папка: {path}")
            print(f"Содержимое: {os.listdir(path)}")
            return path
        else:
            print(f"Папка не найдена: {path}")
    
    return None

if __name__ == "__main__":
    print("=" * 60)
    print("MULTI-LABEL КЛАССИФИКАЦИЯ С BBOX ПАРАМЕТРАМИ")
    print("=" * 60)
    
    data_path = check_data_structure()
    
    if data_path is None:
        print("\nНе удалось найти папку с данными!")
        exit()
    
    print(f"\nИспользуем путь: {data_path}")
    
    try:
        train_dataset, test_dataset, class_names = load_and_split_data(data_path, train_ratio=0.8)
        
        if train_dataset is None:
            exit()
        
        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
        
        print(f"\nИНФОРМАЦИЯ О ДАННЫХ:")
        print(f"Classes: {class_names}")
        print(f"Number of classes: {len(class_names)}")
        print(f"Training samples: {len(train_dataset)}")
        print(f"Test samples: {len(test_dataset)}")
        
        num_classes = len(class_names)
        model = create_resnet_with_bbox(num_classes, pretrained=True)
        
        criterion = MultiLabelLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        yolo_provider = None
        
        print(f"\nНАЧАЛО ОБУЧЕНИЯ MULTI-LABEL МОДЕЛИ С BBOX")
        print("=" * 60)
        
        train_losses, val_accuracies = train_model_with_bbox(
            model, train_loader, test_loader, criterion, optimizer, 
            epochs=5, bbox_provider=yolo_provider
        )
        
        if len(train_losses) > 0:
            plt.figure(figsize=(12, 4))
            
            plt.subplot(1, 2, 1)
            plt.plot(train_losses)
            plt.title('Training Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.grid(True)
            
            plt.subplot(1, 2, 2)
            plt.plot(val_accuracies)
            plt.title('Validation Label Accuracy')
            plt.xlabel('Epoch')
            plt.ylabel('Accuracy (%)')
            plt.grid(True)
            
            plt.tight_layout()
            plt.show()
            
            torch.save({
                'model_state_dict': model.state_dict(),
                'class_names': class_names,
                'model_architecture': 'resnet18_with_bbox',
                'multi_label': True
            }, 'resnet18_with_bbox_multilabel.pth')
            
            print("✅ Multi-label model with BBox saved!")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

def predict_with_bbox(model, image_path, yolo_model_path, class_names, device, transform=None, threshold=0.5):
    """
    Предсказание с использованием YOLO для получения bbox
    """
    if transform is None:
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
    
    yolo_model = YOLO(yolo_model_path)
    
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    yolo_results = yolo_model(image)
    bbox_params = yolo_to_bbox_params(yolo_results[0])
    bbox_params = bbox_params.unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor, bbox_params)
        probabilities = torch.sigmoid(outputs)
        predicted_probs = probabilities.cpu().numpy()[0]
        
        predicted_labels = (predicted_probs > threshold).astype(int)
        predicted_classes = [class_names[j] for j, pred in enumerate(predicted_labels) if pred == 1]
    
    result = {
        'image_path': image_path,
        'predicted_classes': predicted_classes,
        'predicted_probs': predicted_probs,
        'bbox_params': bbox_params.cpu().numpy()[0],
        'class_probabilities': {
            class_name: prob for class_name, prob in zip(class_names, predicted_probs)
        }
    }
    
    print(f"BBox параметры: x_center={bbox_params[0,0]:.3f}, y_center={bbox_params[0,1]:.3f}, "
          f"width={bbox_params[0,2]:.3f}, height={bbox_params[0,3]:.3f}")
    print(f"Предсказанные классы: {', '.join(predicted_classes) if predicted_classes else 'НЕТ'}")
    
    return result

def example_prediction():
    """Пример использования модели для предсказания"""
    try:
        checkpoint = torch.load('resnet18_with_bbox_multilabel.pth', map_location=device)
        class_names = checkpoint['class_names']
        
        model = create_resnet_with_bbox(len(class_names), pretrained=False)
        model.load
        
        _state_dict(checkpoint['model_state_dict'])
        model.to(device)
        model.eval()
        
        print("Модель загружена для предсказания")
        
        # result = predict_with_bbox(
        #     model=model,
        #     image_path="test_image.jpg",
        #     yolo_model_path="path/to/best.pt",
        #     class_names=class_names,
        #     device=device
        # )
        
    except Exception as e:
        print(f"Ошибка при загрузке модели: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import traceback
import cv2
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class MultiLabelImageFolder(Dataset):
    """Кастомный Dataset для multi-label классификации"""
    def __init__(self, data_path, transform=None, label_file=None):
        self.data_path = data_path
        self.transform = transform
        
        self.image_paths = []
        self.labels = []
        self.classes = []
        
        if label_file and os.path.exists(label_file):
            self._load_from_csv(label_file)
        else:
            self._load_from_folders()
    
    def _load_from_folders(self):
        """Загрузка данных из структуры папок (multi-label)"""
        self.classes = sorted([d for d in os.listdir(self.data_path) 
                             if os.path.isdir(os.path.join(self.data_path, d))])
        
        class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        image_dict = {}
        
        for class_idx, class_name in enumerate(self.classes):
            class_path = os.path.join(self.data_path, class_name)
            if not os.path.isdir(class_path):
                continue
                
            for img_name in os.listdir(class_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_path, img_name)
                    
                    if img_path not in image_dict:
                        image_dict[img_path] = []
                    
                    image_dict[img_path].append(class_idx)
        
        self.image_paths = list(image_dict.keys())
        self.labels = [self._to_multi_hot(labels, len(self.classes)) 
                      for labels in image_dict.values()]
        
        print(f"Найдено {len(self.image_paths)} изображений с {len(self.classes)} классами")
    
    def _load_from_csv(self, label_file):
        """Загрузка данных из CSV файла с multi-label метками"""
        df = pd.read_csv(label_file)
        self.classes = [col for col in df.columns if col not in ['image', 'filename']]
        
        for _, row in df.iterrows():
            img_name = row['image'] if 'image' in row else row['filename']
            img_path = os.path.join(self.data_path, img_name)
            
            if os.path.exists(img_path):
                self.image_paths.append(img_path)
                labels = [int(row[cls]) for cls in self.classes]
                self.labels.append(labels)
    
    def _to_multi_hot(self, indices, num_classes):
        """Конвертирует список индексов в multi-hot вектор"""
        multi_hot = [0] * num_classes
        for idx in indices:
            if idx < num_classes:
                multi_hot[idx] = 1
        return multi_hot
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.float32)
        return image, label

def load_and_split_data(data_path, train_ratio=0.8, label_file=None):
    """Загружает данные для multi-label классификации и разделяет на выборки"""
    print(f"Загрузка multi-label данных из: {data_path}")
    
    if not os.path.exists(data_path):
        print(f"Ошибка: Папка {data_path} не существует!")
        return None, None, None
    
    full_dataset = MultiLabelImageFolder(
        data_path=data_path,
        transform=train_transform,
        label_file=label_file
    )
    
    print(f"Найдено классов: {full_dataset.classes}")
    print(f"Всего изображений: {len(full_dataset)}")
    
    train_size = int(train_ratio * len(full_dataset))
    test_size = len(full_dataset) - train_size
    
    print(f"Тренировочные данные: {train_size} изображений")
    print(f"Тестовые данные: {test_size} изображений")
    
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
    
    test_dataset_full = MultiLabelImageFolder(
        data_path=data_path,
        transform=val_transform,
        label_file=label_file
    )
    test_dataset = torch.utils.data.Subset(test_dataset_full, test_dataset.indices)
    
    return train_dataset, test_dataset, full_dataset.classes

class MultiLabelLoss(nn.Module):
    """Loss функция для multi-label классификации"""
    def __init__(self):
        super(MultiLabelLoss, self).__init__()
        self.bce_loss = nn.BCEWithLogitsLoss()
    
    def forward(self, outputs, targets):
        return self.bce_loss(outputs, targets)

def calculate_multi_label_accuracy(outputs, targets, threshold=0.5):
    """Вычисляет accuracy для multi-label классификации"""
    with torch.no_grad():
        preds = torch.sigmoid(outputs) > threshold
        correct = (preds == targets.bool()).float()
        
        exact_match = (correct.sum(dim=1) == targets.size(1)).float().mean()
        label_accuracy = correct.mean()
        
        return exact_match.item(), label_accuracy.item()

class ResNetWithBBox(nn.Module):
    """
    ResNet модель с добавлением параметров bounding box к эмбеддингам
    перед классификационной частью
    """
    def __init__(self, num_classes, pretrained=True, bbox_dim=4):
        super(ResNetWithBBox, self).__init__()
        
        self.resnet = models.resnet18(pretrained=pretrained)
        
        self.original_fc_in_features = self.resnet.fc.in_features
        
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-1])
        
        self.bbox_dim = bbox_dim
        self.classifier = nn.Linear(
            self.original_fc_in_features + bbox_dim, 
            num_classes
        )
        
        nn.init.normal_(self.classifier.weight, 0, 0.01)
        nn.init.constant_(self.classifier.bias, 0)
        
        print(f"Модель инициализирована:")
        print(f"  - Входные признаки ResNet: {self.original_fc_in_features}")
        print(f"  - BBox параметры: {bbox_dim}")
        print(f"  - Выходные классы: {num_classes}")
    
    def forward(self, x, bbox_params=None):
        """
        Forward pass с дополнительными bbox параметрами
        
        Args:
            x: входные изображения [batch_size, 3, H, W]
            bbox_params: тензор bbox параметров [batch_size, 4]
                       формат: [x_center, y_center, width, height] (нормированные 0-1)
        """
        features = self.resnet(x)
        features = features.view(features.size(0), -1)
        
        if bbox_params is None:
            batch_size = features.size(0)
            bbox_params = torch.zeros(batch_size, self.bbox_dim).to(features.device)
        
        combined_features = torch.cat([features, bbox_params], dim=1)
        
        output = self.classifier(combined_features)
        
        return output

def create_resnet_with_bbox(num_classes, pretrained=True):
    """Создает модель ResNet с поддержкой bbox параметров"""
    return ResNetWithBBox(num_classes, pretrained)

def prepare_bbox_params(bbox_tensor, img_size=(224, 224)):
    """
    Подготавливает bbox параметры для модели
    """
    if bbox_tensor.dim() == 1:
        bbox_tensor = bbox_tensor.unsqueeze(0)
    
    img_h, img_w = img_size
    bbox_normalized = bbox_tensor.clone().float()
    
    if bbox_normalized.size(1) == 4:
        x1, y1, w, h = bbox_normalized[:, 0], bbox_normalized[:, 1], bbox_normalized[:, 2], bbox_normalized[:, 3]
        x_center = (x1 + w/2) / img_w
        y_center = (y1 + h/2) / img_h
        width_norm = w / img_w
        height_norm = h / img_h
    else:
        x_center, y_center, width_norm, height_norm = bbox_normalized[:, 0], bbox_normalized[:, 1], bbox_normalized[:, 2], bbox_normalized[:, 3]
        x_center = x_center / img_w
        y_center = y_center / img_h
        width_norm = width_norm / img_w
        height_norm = height_norm / img_h
    
    bbox_params = torch.stack([x_center, y_center, width_norm, height_norm], dim=1)
    bbox_params = torch.clamp(bbox_params, 0.0, 1.0)
    
    return bbox_params

class SimpleBBoxProvider:
    """Простейший провайдер bbox параметров"""
    def __init__(self, img_size=(224, 224)):
        self.img_size = img_size
    
    def __call__(self, images, labels=None):
        batch_size = images.size(0)
        bbox_params_list = []
        
        for i in range(batch_size):
            x_center = 0.5 + random.uniform(-0.2, 0.2)
            y_center = 0.5 + random.uniform(-0.2, 0.2)
            width = random.uniform(0.1, 0.4)
            height = random.uniform(0.1, 0.4)
            
            x_center = max(0.1, min(0.9, x_center))
            y_center = max(0.1, min(0.9, y_center))
            width = max(0.05, min(0.5, width))
            height = max(0.05, min(0.5, height))
            
            bbox_params = torch.tensor([x_center, y_center, width, height]).float()
            bbox_params_list.append(bbox_params)
        
        return torch.stack(bbox_params_list)

def train_model_with_bbox(model, train_loader, val_loader, criterion, optimizer, epochs=25, bbox_provider=None, save_best=True):
    """
    Обучение модели с bbox параметрами и сохранением лучшей модели
    """
    model = model.to(device)
    train_losses = []
    val_accuracies = []
    best_accuracy = 0.0
    best_model_state = None
    
    history = {
        'train_loss': [],
        'val_exact_accuracy': [],
        'val_label_accuracy': [],
        'learning_rate': []
    }
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            if bbox_provider is not None:
                bbox_params = bbox_provider(images, labels)
                bbox_params = bbox_params.to(device)
                outputs = model(images, bbox_params)
            else:
                outputs = model(images)
            
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            
            if batch_idx % 10 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                print(f'Epoch [{epoch+1}/{epochs}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}, LR: {current_lr:.6f}')
        
        model.eval()
        total_exact_accuracy = 0.0
        total_label_accuracy = 0.0
        val_batches = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                if bbox_provider is not None:
                    bbox_params = bbox_provider(images, labels)
                    bbox_params = bbox_params.to(device)
                    outputs = model(images, bbox_params)
                else:
                    outputs = model(images)
                
                exact_acc, label_acc = calculate_multi_label_accuracy(outputs, labels)
                total_exact_accuracy += exact_acc
                total_label_accuracy += label_acc
                val_batches += 1
        
        epoch_loss = running_loss / len(train_loader)
        epoch_exact_accuracy = 100 * total_exact_accuracy / val_batches
        epoch_label_accuracy = 100 * total_label_accuracy / val_batches
        
        train_losses.append(epoch_loss)
        val_accuracies.append(epoch_label_accuracy)
        
        history['train_loss'].append(epoch_loss)
        history['val_exact_accuracy'].append(epoch_exact_accuracy)
        history['val_label_accuracy'].append(epoch_label_accuracy)
        history['learning_rate'].append(optimizer.param_groups[0]['lr'])
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}')
        print(f'Exact Match Accuracy: {epoch_exact_accuracy:.2f}%')
        print(f'Label Accuracy: {epoch_label_accuracy:.2f}%')
        
        if save_best and epoch_label_accuracy > best_accuracy:
            best_accuracy = epoch_label_accuracy
            best_model_state = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict().copy(),
                'optimizer_state_dict': optimizer.state_dict().copy(),
                'accuracy': epoch_label_accuracy,
                'exact_accuracy': epoch_exact_accuracy,
                'loss': epoch_loss
            }
            print(f"НОВЫЙ РЕКОРД! Сохраняем модель с точностью: {best_accuracy:.2f}%")
            
            torch.save({
                'model_state_dict': model.state_dict(),
                'class_names': class_names,
                'model_architecture': 'resnet18_with_bbox',
                'multi_label': True,
                'epoch': epoch + 1,
                'accuracy': best_accuracy,
                'exact_accuracy': epoch_exact_accuracy,
                'loss': epoch_loss
            }, 'best_resnet18_with_bbox_multilabel.pth')
        
        print('-' * 50)
    
    torch.save({
        'model_state_dict': model.state_dict(),
        'class_names': class_names,
        'model_architecture': 'resnet18_with_bbox',
        'multi_label': True,
        'epoch': epochs,
        'accuracy': epoch_label_accuracy,
        'exact_accuracy': epoch_exact_accuracy,
        'loss': epoch_loss
    }, 'last_resnet18_with_bbox_multilabel.pth')
    
    torch.save(history, 'training_history.pth')
    
    print(f"🏆 Лучшая точность за все эпохи: {best_accuracy:.2f}%")
    
    return train_losses, val_accuracies, history, best_accuracy

def check_data_structure():
    """Проверяет существование различных возможных путей"""
    possible_paths = [
        "../merged_augmented_data",
        "merged_augmented_data", 
        "./merged_augmented_data",
        r"C:\Users\user\Desktop\MIPT_year_2\engineer_workshop\Working_directory\Processed_data\merged_augmented_data"
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            print(f"Найдена папка: {path}")
            if os.path.isdir(path):
                contents = os.listdir(path)
                print(f"Содержимое: {contents}")
                class_folders = [item for item in contents if os.path.isdir(os.path.join(path, item))]
                print(f"Найдено папок классов: {len(class_folders)}")
            return path
        else:
            print(f"Папка не найдена: {path}")
    
    return None

def plot_training_history(history, best_accuracy):
    """Визуализация истории обучения"""
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'])
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    
    plt.subplot(1, 3, 2)
    plt.plot(history['val_exact_accuracy'], label='Exact Match Accuracy')
    plt.plot(history['val_label_accuracy'], label='Label Accuracy')
    plt.axhline(y=best_accuracy, color='r', linestyle='--', label=f'Best: {best_accuracy:.1f}%')
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 3, 3)
    plt.plot(history['learning_rate'])
    plt.title('Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

def evaluate_model(model, test_loader, criterion, bbox_provider=None):
    """Оценка модели на тестовом наборе"""
    model.eval()
    test_loss = 0.0
    total_exact_accuracy = 0.0
    total_label_accuracy = 0.0
    test_batches = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            if bbox_provider is not None:
                bbox_params = bbox_provider(images, labels)
                bbox_params = bbox_params.to(device)
                outputs = model(images, bbox_params)
            else:
                outputs = model(images)
            
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            
            exact_acc, label_acc = calculate_multi_label_accuracy(outputs, labels)
            total_exact_accuracy += exact_acc
            total_label_accuracy += label_acc
            test_batches += 1
    
    test_loss /= test_batches
    test_exact_accuracy = 100 * total_exact_accuracy / test_batches
    test_label_accuracy = 100 * total_label_accuracy / test_batches
    
    print(f"\nФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВЫХ ДАННЫХ:")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Exact Match Accuracy: {test_exact_accuracy:.2f}%")
    print(f"Test Label Accuracy: {test_label_accuracy:.2f}%")
    
    return test_loss, test_exact_accuracy, test_label_accuracy

if __name__ == "__main__":
    print("=" * 60)
    print("MULTI-LABEL КЛАССИФИКАЦИЯ С BBOX ПАРАМЕТРАМИ")
    print("=" * 60)
    
    data_path = check_data_structure()
    
    if data_path is None:
        print("\nНе удалось найти папку с данными!")
        exit()
    
    print(f"\nИспользуем путь: {data_path}")
    
    try:
        train_dataset, test_dataset, class_names = load_and_split_data(data_path, train_ratio=0.8)
        
        if train_dataset is None:
            print("Не удалось загрузить данные!")
            exit()
        
        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
        
        print(f"\nИНФОРМАЦИЯ О ДАННЫХ:")
        print(f"Classes: {class_names}")
        print(f"Number of classes: {len(class_names)}")
        print(f"Training samples: {len(train_dataset)}")
        print(f"Test samples: {len(test_dataset)}")
        
        num_classes = len(class_names)
        model = create_resnet_with_bbox(num_classes, pretrained=True)
        
        criterion = MultiLabelLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
        
        bbox_provider = SimpleBBoxProvider()
        
        print(f"\nНАЧАЛО ОБУЧЕНИЯ MULTI-LABEL МОДЕЛИ С BBOX")
        print("=" * 60)
        
        train_losses, val_accuracies, history, best_accuracy = train_model_with_bbox(
            model, train_loader, test_loader, criterion, optimizer, 
            epochs=5, bbox_provider=bbox_provider, save_best=True
        )
        
        plot_training_history(history, best_accuracy)
        
        print("\nЗАГРУЖАЕМ ЛУЧШУЮ МОДЕЛЬ ДЛЯ ФИНАЛЬНОЙ ОЦЕНКИ...")
        checkpoint = torch.load('best_resnet18_with_bbox_multilabel.pth', map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        test_loss, test_exact_accuracy, test_label_accuracy = evaluate_model(
            model, test_loader, criterion, bbox_provider=bbox_provider
        )
        
        print(f"\nОБУЧЕНИЕ ЗАВЕРШЕНО!")
        print(f"Лучшая точность: {best_accuracy:.2f}%")
        print(f"Финальная тестовая точность: {test_label_accuracy:.2f}%")
        print(f"Модели сохранены:")
        print(f"   - best_resnet18_with_bbox_multilabel.pth (лучшая модель)")
        print(f"   - last_resnet18_with_bbox_multilabel.pth (последняя модель)")
        print(f"   - training_history.pth (история обучения)")
        
    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()

def load_best_model(model_path, num_classes, device):
    """Загружает лучшую сохраненную модель"""
    try:
        checkpoint = torch.load(model_path, map_location=device)
        
        model = create_resnet_with_bbox(num_classes, pretrained=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        model.eval()
        
        print(f"Загружена модель с точностью: {checkpoint.get('accuracy', 'N/A')}%")
        print(f"Эпоха: {checkpoint.get('epoch', 'N/A')}")
        
        return model, checkpoint.get('class_names', [])
        
    except Exception as e:
        print(f"Ошибка при загрузке модели: {e}")
        return None, []

def use_trained_model():
    """Пример использования обученной модели"""
    try:
        model, class_names = load_best_model(
            'best_resnet18_with_bbox_multilabel.pth', 
            num_classes=len(class_names), 
            device=device
        )
        
        if model is not None:
            print("Модель готова к использованию!")
            print(f"Классы: {class_names}")
            
            
    except Exception as e:
        print(f"Ошибка: {e}")

if __name__ == "__main__":
    pass